# 🎓 Langfuse Observability & Trace Scoping Guide

This notebook covers the basic concepts of telemetry logs (Traces, Spans, and Generations) and demonstrates how to group multiple operations under a parent trace using the context manager pattern from our **Researcher Lambda**.

## 🚀 What is Langfuse?

**Langfuse** is an open-source **LLM Engineering Platform** focused on observability, metrics, and evaluation. It sits inside your application to capture the exact **Traces, Spans, and Generations** we just mapped out.

### 🛠️ Core Capabilities

* **LLM Observability (Tracing):** It automatically visualizes your agent's nested tree structures. You can instantly see exactly which step (Span) caused a bottleneck or where an LLM call (Generation) failed.
* **Token & Cost Tracking:** It maps the exact input/output tokens from your Generations to real-world pricing models (like OpenAI, Anthropic, or AWS Bedrock), giving you immediate cost breakdowns per trace.
* **Prompt Management:** You can host, version, and manage your prompts directly inside Langfuse, pulling them into your runtime via their SDK without redeploying code.
* **Evaluations & Scoring:** It allows you to run evaluation pipelines (using human feedback, regex, or LLM-as-a-judge) to score responses for quality, hallucination, or toxicity.

## 🕵️‍♂️ Understanding Traces, Spans, and Generations

When monitoring AI applications (like an LLM Agent), we use **Observability** to track what happens under the hood. The system breaks down into three key layers: **Traces**, **Spans**, and **Generations**.

---

### 1. The Definitions
* **Trace:** The entire journey. It tracks a single user request from start to finish.
* **Span:** A single unit of work (a step) inside that journey. A trace is made up of multiple spans.
* **Generation:** A specialized type of span where an LLM actually processes tokens (e.g., text generation, embeddings). **Every Generation is a Span, but not every Span is a Generation.**

---

### 2. Live Agent Example: "Summarize Yesterday's News"
Here is how a single **Trace** looks as it branches out into individual **Spans** and **Generations**:

```text
[TRACE: News Summary Request] (Total Time: 8.5 seconds)
 ├── 🔹 [SPAN 1: Parse User Request] (Duration: 0.2s)
 ├── 🔹 [SPAN 2: Formulate Search Query] -> ⭐ [GENERATION: LLM outputs keywords] (Duration: 1.1s)
 ├── 🔹 [SPAN 3: Execute Web Search Tool] (Duration: 2.0s)
 ├── 🔹 [SPAN 4: Scrape Web Pages] (Duration: 1.5s)
 └── 🔹 [SPAN 5: Generate Final Summary] -> ⭐ [GENERATION: LLM writes the news bullet points] (Duration: 3.7s)

In [1]:
import os
from dotenv import load_dotenv

# Try to load local .env, fallback to the sibling folder's .env if present

load_dotenv("../finai-agent/.env", override=True)

# 1. Langfuse Configuration (defaults)
# These will be loaded from your .env file, or you can uncomment and set them here.
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "sk-lf-...")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "pk-lf-...")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://us.cloud.langfuse.com")

os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST

# 2. OpenAI Dummy Key & Tracing Suppression
# The Agents SDK requires an OpenAI API key to bypass client validation checks.
os.environ["OPENAI_API_KEY"] = "dummy-key-for-agents-sdk"

# We disable the SDK's built-in OpenAI tracing to prevent 401 authentication error warnings
# from printing to the console (since we are logging to Langfuse instead).
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

# 3. AWS Bedrock Config
os.environ.setdefault("BEDROCK_REGION", "us-east-1")
os.environ.setdefault("AWS_REGION_NAME", "us-east-1")

print("✅ Configuration variables loaded!")
print("LANGFUSE_PUBLIC_KEY:", os.getenv("LANGFUSE_PUBLIC_KEY")[:15] if os.getenv("LANGFUSE_PUBLIC_KEY") else None)
print("LANGFUSE_HOST:", os.getenv("LANGFUSE_HOST"))

✅ Configuration variables loaded!
LANGFUSE_PUBLIC_KEY: pk-lf-8221df58-
LANGFUSE_HOST: https://us.cloud.langfuse.com


## Context Manager Tracing (`with trace(...)`)

In production servers like our **Researcher Lambda** ([server.py]), we want multiple operations, tools, and model calls to be grouped together under a single, unified trace.

To do this, we use the `with trace("..."):` context manager provided by the `agents` package.

### Why do we do this?
- The context manager creates a parent **Trace** called `"Researcher"` in Langfuse.
- Under the hood, it sets thread-local/coroutine-local context variables.
- Any LiteLLM calls or helper tools triggered within this block are automatically mapped as nested **Spans** (children) under the parent `"Researcher"` trace, rather than being logged as disconnected standalone events.

In [2]:
import litellm
from agents import Agent, Runner, trace, function_tool
from agents.extensions.models.litellm_model import LitellmModel

# 1. Initialize the LiteLLM model wrapper for the Agents SDK.
agent_model = LitellmModel(model="bedrock/moonshotai.kimi-k2.5")

# 2. Define a simple mock tool.
# We decorate it with @function_tool so the Agents framework can understand its purpose.
@function_tool
def get_inflation_rate(year: int) -> str:
    """
    Retrieves the historical or predicted inflation rate percentage for a given year.
    Use this tool when you need to look up inflation data.
    """
    print(f"🛠️ [Tool Invoked] get_inflation_rate for year: {year}")
    if year == 2025:
        return "3.1%"
    return "2.5%"

# 3. Create the Agent with clear, specific instructions.
agent = Agent(
    name="FinAI Simple Assistant",
    instructions="You are a helpful financial assistant. Retrieve the inflation rate for the requested year using the get_inflation_rate tool and present the result clearly to the user.",
    model=agent_model,
    tools=[get_inflation_rate]
)

# 4. Enable Langfuse callbacks for LiteLLM
litellm.success_callback = ["langfuse"]

print("🧠 Starting Agent execution within trace context...")

# 5. Run the Agent within the trace context manager.
# This groups the Agent execution and the nested Tool execution inside a single trace.
# We pass custom key-value metadata to the trace so it is tagged in the dashboard.
with trace("Researcher", metadata={"env": "genai", "user_group": "students"}):
    try:
        # Pass the agent and input query positionally
        result = await Runner.run(
            agent,
            input="What was the inflation rate in 2025?"
        )
        print("\n📝 Final Agent Output:")
        print(result.final_output)
        
    except Exception as e:
        print(f"\n❌ Trace execution failed: {e}")

# 6. Flush the client queue so the telemetry is sent to Langfuse immediately
from langfuse import Langfuse
Langfuse().flush()
print("\n✅ Trace hierarchy successfully uploaded to Langfuse!")

🧠 Starting Agent execution within trace context...
🛠️ [Tool Invoked] get_inflation_rate for year: 2025

📝 Final Agent Output:
 The inflation rate for 2025 is **3.1%**.

✅ Trace hierarchy successfully uploaded to Langfuse!
